In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from scipy.signal import find_peaks

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "Excel" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from well16_image_matching import TIMESTAMP_OFFSET_SECONDS, run_matching_workflow


In [ ]:
def make_zoom_plot(
    xmin,
    xmax,
    ymin=0,
    ymax=150,
    filename="zoom.png",
    title="Zoomed CH$_4$ Region",
    line_color="tab:blue",
    matched_images_df=None,
):
    fig, ax = plt.subplots(figsize=(8, 4.8))

    ax.plot(df["Elapsed_sec"], df["CH4_ppm"], linewidth=1.8, label="CH$_4$", color=line_color)
    ax.scatter(df["Elapsed_sec"], df["CH4_ppm"], s=14, alpha=0.7, color=line_color)

    if matched_images_df is not None and not matched_images_df.empty:
        region_images = matched_images_df[
            (matched_images_df["Elapsed_sec"] >= xmin)
            & (matched_images_df["Elapsed_sec"] <= xmax)
        ].copy()
        if not region_images.empty:
            ax.scatter(
                region_images["Elapsed_sec"],
                region_images["CH4_ppm"],
                s=52,
                marker="v",
                color="black",
                label="Matched photo",
                zorder=4,
            )

    ax.axhline(
        y=100,
        linestyle="--",
        linewidth=1.5,
        color="red",
        label="100 ppm threshold",
    )
    ax.text(
        xmin + 0.72 * (xmax - xmin),
        102,
        "100 ppm threshold",
        color="red",
        ha="left",
        va="bottom",
        fontsize=10,
    )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    zoom_df = df[(df["Elapsed_sec"] >= xmin) & (df["Elapsed_sec"] <= xmax)].copy()
    peak_indices, _ = find_peaks(
        zoom_df["CH4_ppm"].values,
        height=2.150,
        prominence=0.05,
        distance=1,
    )

    label_points = zoom_df.iloc[peak_indices].copy().sort_values("Elapsed_sec")
    labels_text = []

    for i, (_, row) in enumerate(label_points.iterrows(), start=1):
        key = f"P{i}"
        ax.axvline(
            row["Elapsed_sec"],
            linestyle=":",
            linewidth=1.2,
            color="black",
            alpha=0.8,
        )

        y_text = ymax - 8 - ((i - 1) % 2) * 6
        ax.text(
            row["Elapsed_sec"],
            y_text,
            key,
            ha="center",
            va="bottom",
            fontsize=9,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=0.2),
        )
        labels_text.append(
            f"{key} = ({row['CH4_ppm']:.2f} ppm, {row['Corrected_Datetime'].strftime('%H:%M:%S')})"
        )

    if labels_text:
        ax.text(
            1.02,
            0.98,
            "\n".join(labels_text),
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=9,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.9),
        )

    ax.set_xlabel("Elapsed Time (sec)")
    ax.set_ylabel("CH$_4$ (ppm)")
    ax.set_title(title)
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(frameon=False)

    plt.subplots_adjust(right=0.78)
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
csv_file = PROJECT_ROOT / "Well16.csv"
images_dir = PROJECT_ROOT.parents[2] / "Wells" / "W16" / "Field Visit Data" / "Field Pictures"
output_dir = PROJECT_ROOT / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"CSV file: {csv_file}")
print(f"Images dir: {images_dir}")
print(f"Timestamp correction: +{TIMESTAMP_OFFSET_SECONDS} seconds")


In [ ]:
results = run_matching_workflow(
    csv_path=csv_file,
    image_dir=images_dir,
    output_dir=output_dir,
    offset_seconds=TIMESTAMP_OFFSET_SECONDS,
    tolerance_seconds=1,
)

df = results["gas_df"]
images_df = results["images_df"]
all_matches_df = results["all_matches_df"]
overlap_matches_df = results["overlap_matches_df"]
duplicate_image_times_df = results["duplicate_image_times_df"]
unparsed_images_df = results["unparsed_images_df"]
summary = results["summary"]
output_paths = results["output_paths"]

baseline_ch4 = df.loc[
    (df["Elapsed_sec"] >= 0) & (df["Elapsed_sec"] <= 180),
    "CH4_ppm",
].mean()

summary_df = pd.DataFrame(
    [{"metric": key, "value": value} for key, value in summary.items()]
)
display(summary_df)

print(f"Average CH4 from 0-180 s: {baseline_ch4:.3f} ppm")
print("Output files:")
for name, path in output_paths.items():
    print(f"  {name}: {path}")


In [ ]:
preview_columns = [
    "image_id",
    "image_file",
    "image_datetime",
    "Corrected_Datetime",
    "Elapsed_sec",
    "CH4_ppm",
    "delta_seconds",
    "match_status",
]

display(overlap_matches_df[preview_columns].head(12))

if not duplicate_image_times_df.empty:
    print("Duplicate image timestamps")
    display(duplicate_image_times_df)

if not unparsed_images_df.empty:
    print("Images that did not match the expected filename pattern")
    display(unparsed_images_df.head(10))


In [ ]:
plt.rcParams.update({
    "font.size": 11,
    "axes.linewidth": 1.0,
})

matched_photo_points = overlap_matches_df[
    overlap_matches_df["match_status"] == "matched"
].copy()

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(df["Elapsed_sec"], df["CH4_ppm"], linewidth=1.8, label="CH$_4$", color="#3B5BA5")
ax.scatter(df["Elapsed_sec"], df["CH4_ppm"], s=14, alpha=0.7, color="#3B5BA5")
ax.scatter(
    matched_photo_points["Elapsed_sec"],
    matched_photo_points["CH4_ppm"],
    s=42,
    marker="v",
    color="black",
    label="Matched photos",
    zorder=4,
)

ax.set_xlabel("Elapsed Time (sec)")
ax.set_ylabel("CH$_4$ (ppm)")
ax.set_ylim(0, 150)
ax.set_title("CH$_4$ Concentration vs Time with Matched Photo Timestamps")
ax.grid(True, linestyle="--", alpha=0.3)

ax.axhline(
    y=100,
    linestyle="--",
    linewidth=1.5,
    color="red",
    label="100 ppm threshold",
)
ax.text(
    df["Elapsed_sec"].max() * 0.98,
    102,
    "100 ppm threshold",
    color="red",
    ha="right",
    va="bottom",
    fontsize=10,
)

ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(output_dir / "seconds_vs_ch4_with_photo_matches.png", dpi=300, bbox_inches="tight")
plt.show()

make_zoom_plot(
    220,
    420,
    filename=output_dir / "zoom_region_1.png",
    title="Zoomed Region: 220-420 sec",
    line_color="#4C9F9B",
    matched_images_df=matched_photo_points,
)
make_zoom_plot(
    420,
    980,
    filename=output_dir / "zoom_region_2.png",
    title="Zoomed Region: 420-980 sec",
    line_color="#C98C2B",
    matched_images_df=matched_photo_points,
)
make_zoom_plot(
    980,
    1040,
    filename=output_dir / "zoom_region_3.png",
    title="Zoomed Region: 980-1040 sec",
    line_color="#B94E48",
    matched_images_df=matched_photo_points,
)
make_zoom_plot(
    1210,
    1250,
    filename=output_dir / "zoom_region_4.png",
    title="Zoomed Region: 1210-1250 sec",
    line_color="#8E5C99",
    matched_images_df=matched_photo_points,
)


In [ ]:
top_photo_matches = overlap_matches_df.sort_values("CH4_ppm", ascending=False)[
    [
        "image_id",
        "image_file",
        "image_datetime",
        "Corrected_Datetime",
        "CH4_ppm",
        "Elapsed_sec",
    ]
].head(10)

display(top_photo_matches)
